In [1]:
import ezomero
import tqdm
import shutil
import os
from skimage import io
from ultralytics import YOLO
import matplotlib.pyplot as plt
import numpy as np
import dask.array as da
import tifffile

from sklearn.model_selection import train_test_split
import torch
import glob
import tqdm
torch.cuda.is_available()

True

In [3]:
host = 'omero-nfdi.uni-muenster.de'
group = None
port = 4064
secure = True
user='JohannesSoltwedel'

conn = ezomero.connect(host=host, port=port, secure=secure, user=user, group=group)

In [4]:
dataset_id = 251

In [5]:
working_directory = r'D:\local_data\BiaPoL\yolo_demo_leica\dataset251'  # the directory where you want to save the dataset - replace with your own directory
images_dir = os.path.join(working_directory, 'images')
labels_dir = os.path.join(working_directory, 'labels')
os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

In [6]:
img_ids = ezomero.get_image_ids(conn, dataset=dataset_id)  # replace 259 with the id of the dataset you want to use
img_ids = [i for i in img_ids if len(ezomero.get_roi_ids(conn, image_id=i)) > 0]  # filter images that have rois

In [13]:
image_paths = []
for img_id in tqdm.tqdm(img_ids):

    # get the image and metadata and save the image locally
    metadata, image = ezomero.get_image(conn, image_id=img_id, dim_order='tczyx')
    image_filename = os.path.join(working_directory, 'images', f'{metadata.name}')
    if not image_filename.endswith('.tif'):
        image_filename += '.tif'
    labels_filename = os.path.join(working_directory, 'labels', f'{metadata.name.replace("tif", "txt")}')
    if not labels_filename.endswith('.txt'):
        labels_filename += '.txt'
    
    image = image.squeeze()  # remove singleton dimensions (TCZ) and make pseudo 3-channel
    io.imsave(image_filename, image, check_contrast=False, )
    image_paths.append(image_filename)

    # determine the width and height of the image and get associated rois
    width, height = image.shape[1], image.shape[0]
    roi_ids = ezomero.get_roi_ids(conn, image_id=img_id)

    shapes = []
    for roi_id in roi_ids:
        shape_ids = ezomero.get_shape_ids(conn, roi_id=roi_id)
        for shape_id in shape_ids:
            shapes.append(ezomero.get_shape(conn, shape_id=shape_id))

    # now to convert the shapes to YOLO format
    with open(labels_filename, 'w') as f:
        for shape in shapes:
            w = shape.width / width
            h = shape.height / height

            x = shape.x / width + w / 2
            y = shape.y / height + h / 2

            class_id = 0
            f.write(f'{class_id} {x} {y} {w} {h}\n')

100%|██████████| 86/86 [12:50<00:00,  8.96s/it]


In [14]:
# # load images in batches of 16 images and do batch-normalization to mean=128, std=64
# # then over write the images in the images directory
# batch_size = 16


# image_paths = glob.glob("images/*.tif")

# for i in tqdm.tqdm(range(0, len(image_paths), batch_size)):
#     batch_paths = image_paths[i:i + batch_size]
#     images = [io.imread(p) for p in batch_paths]
#     images = [(img - mean) / std for img in images]  # normalize to mean=128, std=64
#     for j, img in enumerate(images):
#         io.imsave(batch_paths[j], img.astype(np.uint8).squeeze(), check_contrast=False)  # save the image back to disk

In [15]:
# Output directories
output_dirs = {
    'train': {'images': os.path.join(working_directory, 'train/images'), 'labels': os.path.join(working_directory, 'train/labels')},
    'val': {'images': os.path.join(working_directory, 'val/images'), 'labels': os.path.join(working_directory, 'val/labels')},
    'test': {'images': os.path.join(working_directory, 'test/images'), 'labels': os.path.join(working_directory, 'test/labels')}
}

# Create output directories if they don't exist
for key in output_dirs:
    os.makedirs(output_dirs[key]['images'], exist_ok=True)
    os.makedirs(output_dirs[key]['labels'], exist_ok=True)

In [16]:
# Get list of all image files
image_files = [f for f in os.listdir(images_dir) if os.path.isfile(os.path.join(images_dir, f))]

# Split the dataset into train, val, and test
train_files, test_files = train_test_split(image_files, test_size=0.30, random_state=42)
val_files, test_files = train_test_split(test_files, test_size=1/3, random_state=42)  # 1/3 of 30% => 10%

# Move the files to the respective directories
for key, files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
    for file in files:
        image_file = os.path.join(images_dir, file)
        label_file = os.path.join(labels_dir, file.replace('.tif', '.txt'))

        image = io.imread(image_file)
        # normalize to 0..255
        image = (image - image.min()) / (image.max() - image.min()) * 255
        
        #shutil.copy(image_file, output_dirs[key]['images'])
        shutil.copy(label_file, output_dirs[key]['labels'])
        io.imsave(os.path.join(output_dirs[key]['images'], file.replace(".tif", ".png")), image.astype(np.uint8).squeeze(), check_contrast=False)

In [17]:
print('Number of samples in training set:', len(train_files))
print('Number of samples in validation set:', len(val_files))
print('Number of samples in test set:', len(test_files))

Number of samples in training set: 60
Number of samples in validation set: 17
Number of samples in test set: 9


In [23]:
model = YOLO()

In [24]:
model.train(data=os.path.join(working_directory, 'dataset_config.yaml'),
            epochs=200, imgsz=640, device=0, batch=64, flipud=0.5, fliplr=0.5,
            dropout=True, optimizer='Adam', seed=42, plots=True)

New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.23  Python-3.12.11 torch-2.8.0+cu129 CUDA:0 (NVIDIA GeForce RTX 3060 Ti, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\local_data\BiaPoL\yolo_demo_leica\dataset251\dataset_config.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=True, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.9

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001B096BF4560>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480